# DataOps: a new field season arrives

**Does a new field season change how body mass tracks flipper length in penguins?**

You study body mass against flipper length in penguins. How much heavier is a penguin per mm of flipper, within a species? You have the 2007 field season, and the answer is about 40 g/mm. Then the field station sends 2008 and 2009. Your pipeline runs without a complaint, and the answer drops to under 1 g/mm.

Nothing crashed. The data is wrong in a way your pipeline doesn't look for. You'll make it look, roll back to the good result, repair the data on the record, and test that your check can fail. The data is a pet, so nobody edits it by hand. It has versions, and a repair is a recorded decision.

**The short path takes about 15 minutes:** run the setup cells, do step 1, then run `python check.py --short`. Steps 2 and 3 are optional and take about 20 more. Only reading? Each step shows the failure on this page and says how to approach the fix.

Commands that start with `#` are yours. Remove the `#` when you reach them. Everything else runs as it is, once, in order. A cell that starts with `%%bash --no-raise-error` runs as one bash script, and a failed command prints its error and doesn't crash the notebook.

## Get a copy to work in

This page is the untouched notebook, run once. It shows the failures and stops, because the fixes are yours. To do the work, use your own copy.

**[Open it in Binder](https://mybinder.org/v2/gh/ohpcsim/reproducibility-in-ai/binder-02-dataops?urlpath=lab/tree/dataops.ipynb).** It's free and there's nothing to install. The first start can take a few minutes. A Binder session is temporary and has 2 GB of memory, so download anything you want to keep.

**Or run it on your laptop.** This works on Linux and macOS, on x86_64 or ARM, and on Windows through WSL2. You need [Miniforge](https://github.com/conda-forge/miniforge) or any other conda.

```bash
git clone --branch binder-02-dataops --single-branch https://github.com/ohpcsim/reproducibility-in-ai.git dataops
cd dataops
conda env create --file environment.yml
conda activate repro-dataops
jupyter lab dataops.ipynb
```

On Windows, keep the folder in your WSL2 home, not under `/mnt/c`. To start over at any point, run `python scripts/setup_project.py --reset`, which deletes your work.

## The project

This builds `project/`, with three scripts, a cleaning policy, and the 2007 data. It also makes an empty folder, `osn-bucket/`, which stands in for an object storage bucket such as OSN. DVC uses a real bucket the same way, and your keys stay in `--local` config, out of Git (slide 45).

In [ ]:
!python scripts/setup_project.py

In [ ]:
!cd project && git log --format='%h  %s' && ls && cat params.yaml

## Setup: put the data and the pipeline under version control

You only run these cells, but read each one. The commands below are the ones on slide 45. `dvc add` puts a small pointer file in Git and keeps the data out of it. Each `dvc stage add` declares a step, what it reads, and what it writes (slides 28 and 29). `dvc repro` runs whatever is out of date. `dvc push` sends the data to the bucket.

`clean` comes first and `validate` checks its output, so `report` only ever reads data that passed.

In [ ]:
%%bash --no-raise-error
cd project
dvc init
dvc remote add -d storage ../osn-bucket
dvc add --quiet data/penguins.csv
dvc stage add --quiet -n clean -d data/penguins.csv -d clean.py -o results/clean.csv python clean.py
dvc stage add --quiet -n validate -d results/clean.csv -d validate.py -o results/validated.csv python validate.py
dvc stage add --quiet -n report -d results/validated.csv -d report.py -M metrics.json -o results/report.md python report.py

In [ ]:
%%bash --no-raise-error
cd project
dvc repro

In [ ]:
%%bash --no-raise-error
cd project
git add .
git commit --quiet -m "Track the 2007 season and the pipeline"
dvc push
dvc metrics show

## Step 1: a delivery that passes and shouldn't

The field station sends the rest. Update the pointer and rerun, the way you would on a normal day.

In [ ]:
!python scripts/new_season.py

In [ ]:
%%bash --no-raise-error
cd project
dvc add --quiet data/penguins.csv
dvc repro

In [ ]:
%%bash --no-raise-error
cd project
dvc metrics diff

The pipeline ran green, and the slope collapsed. A longer flipper can't mean almost no extra mass. Look at the delivery yourself. `validate.py` checks that the columns exist and that there are rows, which is all it was ever asked to do.

In [ ]:
import pandas as pd

delivery = pd.read_csv("project/data/penguins.csv")
print(delivery[["flipper_length_mm", "body_mass_g"]].describe().round(1))
print("repeated rows:", delivery.duplicated().sum())

Your turn:

1. Add checks to `project/validate.py` until `dvc repro` stops at `validate` on this delivery. Decide what a believable flipper length is, and what counts as a repeated row. Exit with a message a tired colleague can act on, such as `sys.exit("validate: ...")`. `check.py` will plant more than one kind of bad delivery, including a kind this one doesn't have, so think about the `species` column too.
2. Run `dvc repro` and watch it stop. The metrics don't change, because `report` never ran.
3. Roll back. Git restores the pointer, `dvc checkout` restores the data, and `dvc metrics diff` should print nothing. That's how you know you're back. The rollback names the files, because a bare `git checkout .` would also throw away your new `validate.py`.
4. Rerun on the good data and commit your checks.

In [ ]:
%%bash --no-raise-error
cd project
# dvc repro
# git checkout -- data/penguins.csv.dvc dvc.lock metrics.json && dvc checkout
# dvc metrics diff
# dvc repro && git add validate.py dvc.lock && git commit --quiet -m "Validate the range and uniqueness of a delivery" && dvc push

### Check the short path

Run this when step 1 is done. `check.py` clones your last commit into an empty folder, pulls the data from your bucket, and plants its own bad deliveries. Right now it fails, and each line says what to do next. When it passes, a bad delivery stops your pipeline and you've finished the short path. You can stop here.

In [ ]:
!python check.py --short

## Step 2 (optional): a policy nobody declared

Refusing the delivery is right for now, but you still want those two seasons. Most of the rows are good. A person should decide how to repair the rest, and Git should record that decision. That's what the policy in `params.yaml` is for.

There's a problem with it, though. This cell changes the policy, asks DVC what's out of date, and puts the policy back.

In [ ]:
!python scripts/try_policy.py

DVC doesn't see the change. `clean.py` reads `params.yaml`, but `dvc.yaml` doesn't say so. Change the policy and your report is stale, with `dvc repro` telling you everything is up to date. It's the same failure as exercise 1, a number that no longer matches its inputs.

Your turn:

1. Declare the `clean` policy as a dependency of the `clean` stage in `project/dvc.yaml`. A stage can have a `params:` list (the [DVC docs](https://doc.dvc.org/user-guide/project-structure/dvcyaml-files) show the format). Run `try_policy.py` again. It should now report the change.
2. In `project/clean.py`, act on `policy["convert_inches"]` and `policy["drop_duplicates"]`. Leave each repair off unless its setting is on. Don't hand-edit the delivery.
3. Turn both on in `project/params.yaml`. Your `validate.py` stays as strict as it was, and it now checks the repaired data.
4. Take the delivery again, rerun, and compare. You're looking for a slope close to the first season's.
5. Commit and push. Git holds the policy and the pointer, and the bucket holds the data, so you need both.

In [ ]:
%%bash --no-raise-error
# python scripts/try_policy.py
# python scripts/new_season.py
cd project
# dvc add --quiet data/penguins.csv && dvc repro && dvc metrics diff
# git add . && git commit --quiet -m "Accept the 2008 and 2009 seasons, repaired by the recorded policy" && dvc push

## Step 3 (optional): test that the result can move

Slide 47: a pipeline that has never failed on known-wrong input hasn't been tested. Your slope matched, but would it have noticed a change? Change one row of the data, in a way `validate` still accepts. The slope must move in `dvc metrics diff`. Pick your own row and value in place of these.

Save that diff, roll the data back, and commit the record.

In [ ]:
%%bash --no-raise-error
cd project
# python -c "import pandas as pd; d = pd.read_csv('data/penguins.csv'); d.loc[0, 'body_mass_g'] += 500; d.to_csv('data/penguins.csv', index=False)"
# dvc repro && dvc metrics diff | tee negative-control.txt
# git checkout -- data/penguins.csv.dvc dvc.lock metrics.json && dvc checkout && dvc status
# git add negative-control.txt && git commit --quiet -m "Record the negative control"

## Check everything

This is the full check, for steps 1 to 3. It clones your last commit and pulls the data, the way a colleague would (slide 46). Then it plants bad deliveries, changes the policy, and watches what your pipeline does. It works in a throwaway clone, so your project is safe. It takes under a minute.

In [ ]:
!python check.py

## What you've got, and what you don't

A matching slope means the pipeline repeats. Whether flipper length really predicts mass this way is a question for you and your colleagues (slide 50). The tools get you the first, and make the second possible.

## On your own project

Pick the dataset your last result came from. `dvc add` it, write down one thing that would make you refuse a delivery, and put that in a validate stage. Then change one row and see if your result moves. A local folder works as the remote until you have a bucket.

## About the data

Palmer penguins, `data/penguins.csv`, 15 KB, 344 penguins, CC0 1.0. It's unchanged from the [palmerpenguins](https://allisonhorst.github.io/palmerpenguins/) package. `scripts/new_season.py` plants the inches and repeats, and those errors are not in the published data.

Cite it as: Horst AM, Hill AP, Gorman KB (2020). palmerpenguins: Palmer Archipelago (Antarctica) penguin data. R package version 0.1.0. doi:10.5281/zenodo.3960218. The data was first published in Gorman KB, Williams TD, Fraser WR (2014), PLoS ONE 9(3):e90081, doi:10.1371/journal.pone.0090081.